# 合并代码

In [ ]:
import pandas as pd

# 读取数据
# CSV 和 notebook 在同一目录
df = pd.read_csv("../data/裁判文书网数据/2020年裁判文书数据_马克数据网/2020年01月裁判文书数据.csv")
# 查看条数
print("筛选前条数：", len(df))


# 筛选：案件名称包含“交通”
# 搜索“驾驶”是“危险驾驶罪”，一般都是醉驾/酒驾
# na=False 避免案件名称里有空值(NaN)时报错
# mask = df["案件名称"].astype(str).str.contains("交通", na=False)
mask = df["案由"].astype(str).str.contains("交通", na=False)
df_traffic = df[mask].copy()
# 查看筛选结果
print("筛选后条数：", len(df_traffic))


# 筛选：将全文内容为空的筛选出去
# 把全文标准化：NaN -> ""，并去掉首尾空白
fulltext = df_traffic["全文"].fillna("").astype(str).str.strip()
# 过滤条件：不为空、且不是字符串 "nan"/"NaN"
non_empty_mask = fulltext.ne("") & (~fulltext.str.lower().eq("nan"))
df_traffic_nonempty = df_traffic[non_empty_mask].copy()
print("去掉全文为空后条数：", len(df_traffic_nonempty))


# 筛选：只保留某几列
keep_cols = ["案号", "法院", "所属地区", "案件类型", "裁判日期", "当事人", "案由", "全文"]
df_traffic_clean = df_traffic_nonempty.loc[:, [c for c in keep_cols if c in df_traffic_nonempty.columns]].copy()


# 保存
df_traffic_clean.to_csv("../data/裁判文书网数据清洗/2020/2020_01.csv", index=False, encoding="utf-8-sig")

# 批量处理

## 测试读入，读出的路径

In [ ]:
import re
from pathlib import Path
import pandas as pd

IN_ROOT = Path(r"E:\202512LLM交通\data\裁判文书网数据")
OUT_ROOT = Path(r"E:\202512LLM交通\data\裁判文书网数据清洗")

for csv_path in IN_ROOT.rglob("*.csv"):
    # 从父文件夹名提取年份：如 “2020年裁判文书数据_马克数据网”
    m_year = re.search(r"(\d{4})年", csv_path.parent.name)
    year = m_year.group(1) if m_year else "unknown"

    # 输出目录：.../裁判文书网数据清洗/2020/
    out_dir = OUT_ROOT / year
    out_dir.mkdir(parents=True, exist_ok=True)

    # 输出文件名：优先 YYYY_MM.csv（从文件名提取“XX月”），否则用原文件名
    m_month = re.search(r"(\d{2})月", csv_path.name)
    out_name = f"{year}_{m_month.group(1)}.csv" if m_month else csv_path.name
    out_path = out_dir / out_name

    print('input:',csv_path,'\noutput:',out_path)
    # 读取数据（可选：加个编码兜底）
    

## 处理

In [ ]:
import re
from pathlib import Path
import pandas as pd

IN_ROOT = Path(r"E:\202512LLM交通\data\1.裁判文书网数据")
OUT_ROOT = Path(r"E:\202512LLM交通\data\2.裁判文书网数据清洗")

for csv_path in IN_ROOT.rglob("*.csv"):
    # 从父文件夹名提取年份：如 “2020年裁判文书数据_马克数据网”
    m_year = re.search(r"(\d{4})年", csv_path.parent.name)
    year = m_year.group(1) if m_year else "unknown"

    # 输出目录：.../裁判文书网数据清洗/2020/
    out_dir = OUT_ROOT / year
    out_dir.mkdir(parents=True, exist_ok=True)

    # 输出文件名：优先 YYYY_MM.csv（从文件名提取“XX月”），否则用原文件名
    m_month = re.search(r"(\d{2})月", csv_path.name)
    out_name = f"{year}_{m_month.group(1)}.csv" if m_month else csv_path.name
    out_path = out_dir / out_name

    # 读取数据（可选：加个编码兜底）
    try:
        df = pd.read_csv(csv_path)
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="gb18030")

    
    # ===================== 下面处理代码不要动（仅整体缩进） =====================
    # 查看条数
    print(out_name,"筛选前条数：", len(df))

    # 筛选：案件名称包含“交通”
    # 搜索“驾驶”是“危险驾驶罪”，一般都是醉驾/酒驾
    # na=False 避免案件名称里有空值(NaN)时报错
    # mask = df["案件名称"].astype(str).str.contains("交通", na=False)
    mask = df["案由"].astype(str).str.contains("交通", na=False)
    df_traffic = df[mask].copy()
    # 查看筛选结果
    print(out_name,"筛选后条数：", len(df_traffic))

    # 筛选：将全文内容为空的筛选出去
    # 把全文标准化：NaN -> ""，并去掉首尾空白
    fulltext = df_traffic["全文"].fillna("").astype(str).str.strip()
    # 过滤条件：不为空、且不是字符串 "nan"/"NaN"
    non_empty_mask = fulltext.ne("") & (~fulltext.str.lower().eq("nan"))
    df_traffic_nonempty = df_traffic[non_empty_mask].copy()
    print(out_name,"去掉全文为空后条数：", len(df_traffic_nonempty))

    # 筛选：只保留某几列
    keep_cols = ["案号", "法院", "所属地区", "案件类型", "裁判日期", "当事人", "案由", "全文"]
    df_traffic_clean = df_traffic_nonempty.loc[:, [c for c in keep_cols if c in df_traffic_nonempty.columns]].copy()

    # 保存（改为动态 out_path）
    df_traffic_clean.to_csv(out_path, index=False, encoding="utf-8-sig")


## 合并所有csv

In [ ]:
import os, glob, csv, sys

root = r"E:\202512LLM交通\data\2.裁判文书网数据清洗"
out  = r"E:\202512LLM交通\data\3.数据合并\裁判文书网2000-2021.csv"
os.makedirs(os.path.dirname(out), exist_ok=True)

csv.field_size_limit(sys.maxsize)  # 全文字段很长时避免报错

files = []
for y in range(2000, 2022):  # 2000-2021
    files += sorted(glob.glob(os.path.join(root, str(y), "*.csv")))

first = True
with open(out, "w", newline="", encoding="utf-8-sig") as fw:
    w = csv.writer(fw)
    for f in files:
        print(f)
        with open(f, "r", newline="", encoding="utf-8-sig") as fr:
            r = csv.reader(fr)
            header = next(r, None)
            if first and header:
                w.writerow(header)   # 只写一次表头
                first = False
            for row in r:
                if row:
                    w.writerow(row)

print("merged", len(files), "files ->", out)


## 查看行数和前几行

In [ ]:
import pandas as pd

fp = r"E:\202512LLM交通\data\3.数据合并\裁判文书网2000-2021.csv"

head5 = pd.read_csv(fp, encoding="utf-8-sig", nrows=5)
rows = sum(1 for _ in open(fp, "r", encoding="utf-8-sig")) - 1  # 减去表头

print("行数：", rows)
print(head5)


## 取最后1w行

In [ ]:
import csv, sys
from collections import deque

inp = r"E:\202512LLM交通\data\3.数据合并\裁判文书网2000-2021.csv"
out = r"E:\202512LLM交通\data\3.数据合并\裁判文书网2000-2021_last1w.csv"

csv.field_size_limit(sys.maxsize)

buf = deque(maxlen=10000)

with open(inp, "r", newline="", encoding="utf-8-sig") as fr:
    r = csv.reader(fr)
    header = next(r)
    for row in r:
        buf.append(row)

with open(out, "w", newline="", encoding="utf-8-sig") as fw:
    w = csv.writer(fw)
    w.writerow(header)
    w.writerows(buf)

print("saved:", len(buf), "rows ->", out)


# 将csv分成2个文件

In [ ]:
import csv
import os
import sys

# ====== 需要你修改的参数 ======
input_path = r"E:\202512LLM交通\data\3.数据合并/裁判文书网2000-2021.csv"          # 原始CSV路径
output_path_1 = r"E:/202512LLM交通/data/3.数据合并/裁判文书网2000-2021_1_2.csv"
output_path_2 = r"E:/202512LLM交通/data/3.数据合并/裁判文书网2000-2021_2_2.csv"
encoding = "utf-8"                 # 常见：utf-8 / utf-8-sig / gbk
# ============================

if not os.path.isfile(input_path):
    raise FileNotFoundError(f"找不到输入文件: {input_path}")

# 关键：放大csv字段大小限制（解决 field larger than field limit）
max_int = sys.maxsize
while True:
    try:
        csv.field_size_limit(max_int)
        break
    except OverflowError:
        max_int = int(max_int / 10)

print(f"csv.field_size_limit 已设置为: {csv.field_size_limit()}")

# 1) 统计总数据行数（不含表头）
# 注意：这里按“物理行”统计，若你的CSV字段里含有换行且被引号包住，会导致统计不准；
# 这种情况需要改成用 csv.reader 统计（也可以，我后面可给你版本）。
with open(input_path, "r", encoding=encoding, newline="") as f:
    total_lines_including_header = sum(1 for _ in f)

if total_lines_including_header <= 1:
    raise ValueError("CSV文件只有表头或为空，无法分割。")

total_data_rows = total_lines_including_header - 1
first_part_rows = total_data_rows // 2  # 第一份数据行数（对半，奇数则第二份多1）

print(f"总数据行数: {total_data_rows}")
print(f"第一份: {first_part_rows} 行数据，第二份: {total_data_rows - first_part_rows} 行数据")

# 2) 重新读一遍，用 csv.reader 正确解析并写入两个文件（都写表头）
with open(input_path, "r", encoding=encoding, newline="") as fin, \
     open(output_path_1, "w", encoding=encoding, newline="") as fout1, \
     open(output_path_2, "w", encoding=encoding, newline="") as fout2:

    reader = csv.reader(fin)
    header = next(reader)

    writer1 = csv.writer(fout1)
    writer2 = csv.writer(fout2)

    writer1.writerow(header)
    writer2.writerow(header)

    count = 0
    for row in reader:
        if count < first_part_rows:
            writer1.writerow(row)
        else:
            writer2.writerow(row)
        count += 1

print("分割完成：")
print(f"  -> {output_path_1}")
print(f"  -> {output_path_2}")


# 确认分割行数成功

In [ ]:
import pandas as pd

fp1 = r"E:/202512LLM交通/data/3.数据合并/裁判文书网2000-2021_1_2.csv"

rows1 = sum(1 for _ in open(fp1, "r", encoding="utf-8-sig")) - 1  # 减去表头
head1 = pd.read_csv(fp1, encoding="utf-8-sig", nrows=5)

print("行数：", rows1)
print("前5行：", head1)

In [ ]:
import pandas as pd

fp2 = r"E:/202512LLM交通/data/3.数据合并/裁判文书网2000-2021_2_2.csv"

rows2 = sum(1 for _ in open(fp1, "r", encoding="utf-8-sig")) - 1  # 减去表头
head2 = pd.read_csv(fp2, encoding="utf-8-sig", nrows=5)

print("行数：", rows2)
print("前5行：", head2)